In [1]:
# =================================================================================================
# NHANES 2013-2014
# Medical Conditions Questionnaire
#
# File:
#   MCQ_H.xpt
#
# Component:
#   Questionnaire - Medical Conditions
#
# Purpose:
#
#   Convert NHANES MCQ_H XPT -> fidelity-preserving corrected CSV.
#
#
# ATTACHED SOURCE FINDINGS
# -------------------------------------------------------------------------------------------------
#
#   Rows                         = 9,770
#   Columns                      = 95
#
#   SEQN range                   = 73557-83731
#   Missing SEQN                 = 0
#   Duplicate SEQN               = 0
#
#   SAS XPORT tiny-zero values   = 991
#
#   Tiny-zero variables:
#
#       MCQ180B       2
#       MCQ180C       3
#       MCQ180K       4
#       MCQ206        1
#       MCQ380      981
#
#       TOTAL       991
#
#   Ordinary numeric zeros before repair = 0
#
#   Genuine decimal variables            = NONE
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
#   1. Restore ONLY validated SAS XPORT tiny-zero artifacts.
#
#   2. Do NOT globally replace tiny values in arbitrary NHANES files.
#
#   3. Remove unnecessary ".0" only from proven whole-number/code fields.
#
#   4. Preserve missing positions exactly.
#
#   5. Do NOT round.
#
#   6. Do NOT use float_format=.
#
#   7. Validate CDC frequencies/ranges.
#
#   8. Validate XPT -> CSV round-trip numeric fidelity.
#
#
# GENERAL NHANES PIPELINE RULE
# -------------------------------------------------------------------------------------------------
#
#   Genuine Dietary / Laboratory / Examination / Demographic decimal
#   variables must remain AS IS.
#
#   This particular MCQ_H file contains no genuine decimal variables.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "MCQ_H.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "MCQ_H_corrected.csv"
)


# =================================================================================================
# 3. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched pandas-decoded source.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2013-2014 MCQ_H XPT -> CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


# =================================================================================================
# 4. EXPECTED STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "MCQ010",
    "MCQ025",
    "MCQ035",
    "MCQ040",
    "MCQ050",
    "AGQ030",
    "MCQ053",
    "MCQ070",
    "MCQ075",
    "MCQ080",
    "MCQ082",
    "MCQ084",
    "MCQ086",
    "MCQ092",
    "MCD093",
    "MCQ149",
    "MCQ151",

    "MCQ160A",
    "MCQ180A",
    "MCQ195",

    "MCQ160N",
    "MCQ180N",

    "MCQ160B",
    "MCQ180B",

    "MCQ160C",
    "MCQ180C",

    "MCQ160D",
    "MCQ180D",

    "MCQ160E",
    "MCQ180E",

    "MCQ160F",
    "MCQ180F",

    "MCQ160G",
    "MCQ180G",

    "MCQ160M",
    "MCQ170M",
    "MCQ180M",

    "MCQ160K",
    "MCQ170K",
    "MCQ180K",

    "MCQ160L",
    "MCQ170L",
    "MCQ180L",

    "MCQ160O",

    "MCQ203",
    "MCQ206",

    "MCQ220",

    "MCQ230A",
    "MCQ230B",
    "MCQ230C",
    "MCQ230D",

    "MCQ240A",
    "MCQ240AA",
    "MCQ240B",
    "MCQ240BB",
    "MCQ240C",
    "MCQ240CC",
    "MCQ240D",
    "MCQ240DD",
    "MCQ240DK",
    "MCQ240E",
    "MCQ240F",
    "MCQ240G",
    "MCQ240H",
    "MCQ240I",
    "MCQ240J",
    "MCQ240K",
    "MCQ240L",
    "MCQ240M",
    "MCQ240N",
    "MCQ240O",
    "MCQ240P",
    "MCQ240Q",
    "MCQ240R",
    "MCQ240S",
    "MCQ240T",
    "MCQ240U",
    "MCQ240V",
    "MCQ240W",
    "MCQ240X",
    "MCQ240Y",
    "MCQ240Z",

    "MCQ300A",
    "MCQ300B",
    "MCQ300C",

    "MCQ365A",
    "MCQ365B",
    "MCQ365C",
    "MCQ365D",

    "MCQ370A",
    "MCQ370B",
    "MCQ370C",
    "MCQ370D",

    "MCQ380"

]


EXPECTED_ROWS = 9770


if (
    df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "MCQ_H column structure differs from validated attachment.\n\n"
        f"Expected columns: {len(EXPECTED_COLUMNS)}\n"
        f"Actual columns:   {len(df.columns)}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected MCQ_H row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: Structure = 9,770 rows x 95 columns."
)


# =================================================================================================
# 5. VERIFY ALL FIELDS LOADED NUMERICALLY
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 95 MCQ_H variables "
        "to load numerically from XPT."
    )


print(
    "PASS: All 95 variables loaded numerically."
)


# =================================================================================================
# 6. SAS XPORT TINY-ZERO ARTIFACT
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


ordinary_zero_counts = (

    df[EXPECTED_COLUMNS]

    .eq(0)

    .sum()

)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)


print(
    "Ordinary numeric zeros before repair:",
    f"{ordinary_zero_total:,}"
)


print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(

    tiny_counts[
        tiny_counts > 0
    ]

    .to_string()

)


# =================================================================================================
# 7. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

EXPECTED_TINY_COUNTS = {

    "MCQ180B": 2,

    "MCQ180C": 3,

    "MCQ180K": 4,

    "MCQ206": 1,

    "MCQ380": 981

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: MCQ_H tiny-value pattern differs "
        "from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 991:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total:,}"
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed before correction."
    )


print(
    "PASS: Exact 991-value tiny-zero pattern validated."
)


# =================================================================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# =================================================================================================

#
# CDC confirms zero is legitimate for:
#
#   MCQ180B  age when heart failure first diagnosed
#   MCQ180C  age when coronary heart disease first diagnosed
#   MCQ180K  age when chronic bronchitis first diagnosed
#   MCQ206   age when first told had jaundice
#   MCQ380   Never had trouble remembering
#
#
# DO NOT apply this replacement globally to arbitrary variables.
#

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[
        col
    ] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "\nTotal restored:",
    f"{total_corrections:,}"
)


if total_corrections != 991:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "PASS: 991 validated zeros restored."
)


# =================================================================================================
# 9. BUILD EXPECTED CORRECTED SOURCE
# =================================================================================================

#
# This dataframe represents exactly what the scientifically
# corrected source should contain:
#
# original XPT
# +
# only the five validated tiny-zero replacements.
#

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# =================================================================================================
# 10. VERIFY NO TINY VALUES REMAIN
# =================================================================================================

remaining_tiny = int(

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

    .sum()

)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No XPORT tiny-zero artifacts remain."
)


# =================================================================================================
# 11. VERIFY FINAL ZERO PATTERN
# =================================================================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if count > 0:

        final_zero_counts[
            col
        ] = count


print(
    "\n--- CORRECTED ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Corrected zero pattern differs from expected.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Corrected dataframe contains exactly "
    "991 legitimate zeros."
)


# =================================================================================================
# 12. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() does NOT change actual values.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        fractional_count = 0


    else:

        fractional_count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = fractional_count


fractional_variables = {

    col: count

    for col, count in fractional_counts.items()

    if count > 0

}


print(
    "Variables with genuine fractional values:",
    fractional_variables
)


if fractional_variables:

    raise ValueError(
        "Unexpected genuine decimal variables found:\n"
        + str(
            fractional_variables
        )
    )


print(
    "PASS: MCQ_H contains no genuine decimal variables."
)


# =================================================================================================
# 13. FIELD TYPE PLAN
# =================================================================================================

#
# Every nonmissing value is mathematically integral after the
# validated zero restoration.
#

INTEGER_COLUMNS = (
    EXPECTED_COLUMNS.copy()
)


DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 14. SEQN SOURCE VALIDATION
# =================================================================================================

seqn_min = int(
    df["SEQN"]
    .min()
)


seqn_max = int(
    df["SEQN"]
    .max()
)


seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicates = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


print(
    "\n--- SEQN CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicates
)


if seqn_min != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 73557-83731 validated."
)


# =================================================================================================
# 15. CDC CHECK - MCQ180B
# =================================================================================================

#
# CDC:
#
#   0-79 years       = 166
#   80 years or more = 15
#   77777 Refused    = 0
#   99999 Don't know = 1
#   Missing          = 9588
#

mcq180b_0_79 = int(

    df["MCQ180B"]

    .between(
        0,
        79
    )

    .sum()

)


mcq180b_80 = int(
    df["MCQ180B"]
    .eq(80)
    .sum()
)


mcq180b_99999 = int(
    df["MCQ180B"]
    .eq(99999)
    .sum()
)


mcq180b_missing = int(
    df["MCQ180B"]
    .isna()
    .sum()
)


if mcq180b_0_79 != 166:

    raise ValueError(
        "MCQ180B 0-79 count differs from CDC."
    )


if mcq180b_80 != 15:

    raise ValueError(
        "MCQ180B=80 differs from CDC."
    )


if mcq180b_99999 != 1:

    raise ValueError(
        "MCQ180B Don't know count differs from CDC."
    )


if mcq180b_missing != 9588:

    raise ValueError(
        "MCQ180B missing count differs from CDC."
    )


print(
    "PASS: MCQ180B matches CDC."
)


# =================================================================================================
# 16. CDC CHECK - MCQ180C
# =================================================================================================

#
# CDC:
#
#   0-79 years       = 210
#   80+              = 21
#   99999 Don't know = 1
#   Missing          = 9538
#

if int(

    df["MCQ180C"]
    .between(0, 79)
    .sum()

) != 210:

    raise ValueError(
        "MCQ180C 0-79 count differs from CDC."
    )


if int(
    df["MCQ180C"]
    .eq(80)
    .sum()
) != 21:

    raise ValueError(
        "MCQ180C=80 differs from CDC."
    )


if int(
    df["MCQ180C"]
    .eq(99999)
    .sum()
) != 1:

    raise ValueError(
        "MCQ180C Don't know count differs from CDC."
    )


if int(
    df["MCQ180C"]
    .isna()
    .sum()
) != 9538:

    raise ValueError(
        "MCQ180C missing count differs from CDC."
    )


print(
    "PASS: MCQ180C matches CDC."
)


# =================================================================================================
# 17. CDC CHECK - MCQ180K
# =================================================================================================

#
# CDC:
#
#   0-79 years       = 304
#   80+              = 6
#   99999 Don't know = 10
#   Missing          = 9450
#

if int(

    df["MCQ180K"]
    .between(0, 79)
    .sum()

) != 304:

    raise ValueError(
        "MCQ180K 0-79 count differs from CDC."
    )


if int(
    df["MCQ180K"]
    .eq(80)
    .sum()
) != 6:

    raise ValueError(
        "MCQ180K=80 differs from CDC."
    )


if int(
    df["MCQ180K"]
    .eq(99999)
    .sum()
) != 10:

    raise ValueError(
        "MCQ180K Don't know count differs from CDC."
    )


if int(
    df["MCQ180K"]
    .isna()
    .sum()
) != 9450:

    raise ValueError(
        "MCQ180K missing count differs from CDC."
    )


print(
    "PASS: MCQ180K matches CDC."
)


# =================================================================================================
# 18. CDC CHECK - MCQ206
# =================================================================================================

#
# CDC:
#
#   0-72 years      = 155
#   80+             = 0
#   999 Don't know  = 2
#   Missing         = 9613
#
# CDC explicitly instructs:
#
#   IF LESS THAN 1 YEAR, ENTER 0
#

if int(

    df["MCQ206"]

    .between(
        0,
        72
    )

    .sum()

) != 155:

    raise ValueError(
        "MCQ206 0-72 count differs from CDC."
    )


if int(
    df["MCQ206"]
    .eq(999)
    .sum()
) != 2:

    raise ValueError(
        "MCQ206 Don't know count differs from CDC."
    )


if int(
    df["MCQ206"]
    .isna()
    .sum()
) != 9613:

    raise ValueError(
        "MCQ206 missing count differs from CDC."
    )


print(
    "PASS: MCQ206 matches CDC."
)


# =================================================================================================
# 19. CDC CHECK - MCQ380
# =================================================================================================

#
# CDC:
#
#   0 = Never                    981
#   1 = About once               445
#   2 = Two or three times       279
#   3 = Nearly every day          84
#   4 = Several times a day       47
#   7 = Refused                    0
#   9 = Don't know                 4
#   Missing                     7930
#

MCQ380_EXPECTED = {

    0.0: 981,

    1.0: 445,

    2.0: 279,

    3.0: 84,

    4.0: 47,

    9.0: 4

}


mcq380_actual = (

    df["MCQ380"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


mcq380_missing = int(

    df["MCQ380"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: MCQ380 ---"
)


print(
    mcq380_actual
)


print(
    "Missing:",
    mcq380_missing
)


if mcq380_actual != MCQ380_EXPECTED:

    raise ValueError(
        "MCQ380 distribution differs from CDC.\n\n"
        f"Expected:\n{MCQ380_EXPECTED}\n\n"
        f"Actual:\n{mcq380_actual}"
    )


if mcq380_missing != 7930:

    raise ValueError(
        "MCQ380 missing count differs from CDC."
    )


print(
    "PASS: MCQ380 matches CDC exactly."
)


# =================================================================================================
# 20. CDC CHECK - MCQ220 CANCER VARIABLE
# =================================================================================================

#
# Important for your analysis.
#
# CDC:
#
#   1 = Yes           547
#   2 = No           5222
#   7 = Refused         0
#   9 = Don't know      0
#   Missing          4001
#

MCQ220_EXPECTED = {

    1.0: 547,

    2.0: 5222

}


mcq220_actual = (

    df["MCQ220"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


mcq220_missing = int(

    df["MCQ220"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: MCQ220 ---"
)


print(
    mcq220_actual
)


print(
    "Missing:",
    mcq220_missing
)


if mcq220_actual != MCQ220_EXPECTED:

    raise ValueError(
        "MCQ220 distribution differs from CDC.\n\n"
        f"Expected:\n{MCQ220_EXPECTED}\n\n"
        f"Actual:\n{mcq220_actual}"
    )


if mcq220_missing != 4001:

    raise ValueError(
        "MCQ220 missing count differs from CDC."
    )


print(
    "PASS: MCQ220 matches CDC exactly."
)


# =================================================================================================
# 21. VERIFY ONLY VALIDATED SOURCE VALUES CHANGED
# =================================================================================================

pre_conversion_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_conversion_failures.append(
            col
        )


if pre_conversion_failures:

    raise ValueError(
        "Unexpected values changed during correction:\n"
        + str(
            pre_conversion_failures
        )
    )


print(
    "PASS: Only the validated 991 tiny-zero values changed."
)


# =================================================================================================
# 22. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    original_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    corrected_missing = (

        df[col]

        .isna()

        .to_numpy()

    )


    missing_position_errors += int(

        (
            original_missing
            !=
            corrected_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed during correction."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 23. CONVERT PROVEN WHOLE-NUMBER VARIABLES TO NULLABLE Int64
# =================================================================================================

#
# This changes:
#
#     SEQN = 73557.0
#
# to:
#
#     SEQN = 73557
#
#
# and:
#
#     MCQ220 = 1.0
#
# to:
#
#     MCQ220 = 1
#
#
# This is safe because every nonmissing field has already been
# verified to contain whole-number values only.
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: All 95 proven integral fields converted to Int64."
)


# =================================================================================================
# 24. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 25. EXPORT CORRECTED CSV
# =================================================================================================

#
# IMPORTANT:
#
# No rounding.
#
# No float_format=.
#
# This same export rule protects genuine decimal variables
# elsewhere in your NHANES pipeline.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 26. RE-READ CSV USING ROUND-TRIP PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 27. XPT -> CSV NUMERIC ROUND-TRIP FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All 95 fields round-trip exactly."
)


# =================================================================================================
# 28. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 29. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' formatting."
)


# =================================================================================================
# 30. FINAL SEQN CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN contains '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV SEQN contains invalid formatting."
    )


seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if seqn_numeric.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if seqn_numeric.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN is clean."
)


# =================================================================================================
# 31. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 32. FINAL ZERO PATTERN CHECK
# =================================================================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    zero_count = int(

        values
        .eq(0)
        .sum()

    )


    if zero_count > 0:

        csv_zero_counts[
            col
        ] = zero_count


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "991 validated true zeros."
)


# =================================================================================================
# 33. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_notation_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_errors[
            col
        ] = count


if scientific_notation_errors:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_errors
        )
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 34. FINAL MISSING-POSITION CHECK
# =================================================================================================

final_missing_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    csv_missing = (

        raw_csv[col]

        .eq("")

        .to_numpy()

    )


    final_missing_errors += int(

        (
            source_missing
            !=
            csv_missing
        )

        .sum()

    )


if final_missing_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from source XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 35. FINAL MCQ220 RECHECK
# =================================================================================================

final_mcq220 = (

    raw_csv["MCQ220"]

    .value_counts()

    .to_dict()

)


if final_mcq220 != {

    "1": 547,

    "2": 5222,

    "": 4001

}:

    raise ValueError(
        "Final MCQ220 distribution differs."
    )


print(
    "PASS: Final MCQ220:"
)


print(
    "  Yes     = 547"
)


print(
    "  No      = 5,222"
)


print(
    "  Missing = 4,001"
)


# =================================================================================================
# 36. FINAL MCQ380 RECHECK
# =================================================================================================

final_mcq380 = (

    raw_csv["MCQ380"]

    .value_counts()

    .to_dict()

)


EXPECTED_FINAL_MCQ380 = {

    "0": 981,

    "1": 445,

    "2": 279,

    "3": 84,

    "4": 47,

    "9": 4,

    "": 7930

}


if final_mcq380 != EXPECTED_FINAL_MCQ380:

    raise ValueError(
        "Final MCQ380 distribution differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_MCQ380}\n\n"
        f"Actual:\n{final_mcq380}"
    )


print(
    "PASS: Final MCQ380 matches CDC."
)


# =================================================================================================
# 37. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2013-2014 MCQ_H CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2013-2014"
)


print(
    ""
)


print(
    "Component:"
)


print(
    "  Questionnaire - Medical Conditions"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    ""
)


print(
    "Original tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables with tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Validated zero restorations:",
    f"{total_corrections:,}"
)


print(
    ""
)


print(
    "Zero-restoration pattern:"
)


print(
    "  MCQ180B = 2"
)


print(
    "  MCQ180C = 3"
)


print(
    "  MCQ180K = 4"
)


print(
    "  MCQ206  = 1"
)


print(
    "  MCQ380  = 981"
)


print(
    ""
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    ""
)


print(
    "MCQ220:"
)


print(
    "  Yes     = 547"
)


print(
    "  No      = 5,222"
)


print(
    "  Missing = 4,001"
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Remaining tiny artifacts: 0"
)


print(
    "Unexpected source-value changes: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Round-trip numeric failures: 0"
)


print(
    "=" * 110
)

NHANES 2013-2014 MCQ_H XPT -> CSV
Rows: 9,770
Columns: 95
PASS: Structure = 9,770 rows x 95 columns.
PASS: All 95 variables loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT values: 991

Tiny values by variable:
MCQ180B      2
MCQ180C      3
MCQ180K      4
MCQ206       1
MCQ380     981
PASS: Exact 991-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
MCQ180B: 2
MCQ180C: 3
MCQ180K: 4
MCQ206: 1
MCQ380: 981

Total restored: 991
PASS: 991 validated zeros restored.
PASS: No XPORT tiny-zero artifacts remain.

--- CORRECTED ZERO PATTERN ---
{'MCQ180B': 2, 'MCQ180C': 3, 'MCQ180K': 4, 'MCQ206': 1, 'MCQ380': 981}
PASS: Corrected dataframe contains exactly 991 legitimate zeros.

--- FRACTIONAL VALUE CHECK ---
Variables with genuine fractional values: {}
PASS: MCQ_H contains no genuine decimal variables.

--- FIELD TYPE PLAN ---
Whole-number/code fields: 95
Genuine decimal fields: []

--- SEQN CHECK ---
Minimum: 73557
Maximum: 83731
